In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======================================================
# CONFIG
# ======================================================
RESULTS_DIR = "../data/CDG-FCO/RESULTS"
PHASES = ["take_off", "cruising", "landing"]

# Order matters for readability
PIPELINE_ORDER = [
    "KALMAN",
    "ARIMA",
    "LSTM",
    "BILSTM",
    "MEAN_SIMILAR_FLIGHT",
    "RAG"
]


# ======================================================
# LOAD ALL RESULTS
# ======================================================
def load_all_results(results_dir):
    all_results = {}

    for fname in os.listdir(results_dir):
        if not fname.endswith(".json"):
            continue

        with open(os.path.join(results_dir, fname), "r", encoding="utf-8") as f:
            data = json.load(f)

        for pipeline_name, pipeline_data in data.items():
            all_results[pipeline_name] = pipeline_data

    return all_results


results = load_all_results(RESULTS_DIR)

# Keep consistent order
pipelines = [p for p in PIPELINE_ORDER if p in results]


# ======================================================
# BUILD DATAFRAME
# ======================================================
rows = []

for pipeline in pipelines:
    for phase in PHASES:
        values = results[pipeline][phase]

        for v in values:
            rows.append({
                "pipeline": pipeline,
                "phase": phase,
                "error_km": v
            })

df = pd.DataFrame(rows)


# ======================================================
# SUMMARY STATISTICS TABLE
# ======================================================
stats_table = (
    df.groupby(["pipeline", "phase"])["error_km"]
      .agg(["mean", "median", "std", "min", "max"])
      .round(2)
)

print("\n===== SUMMARY STATISTICS =====")
print(stats_table)


# ======================================================
# BAR PLOT — MEAN ERROR PER PIPELINE
# ======================================================
for phase in PHASES:
    phase_df = df[df["phase"] == phase]

    means = (
        phase_df.groupby("pipeline")["error_km"]
        .mean()
        .loc[pipelines]
    )

    plt.figure(figsize=(10, 5))
    means.plot(kind="bar")
    plt.ylabel("Mean Haversine Error (km)")
    plt.title(f"Mean Haversine Error — {phase}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ======================================================
# BOXPLOT — DISTRIBUTION PER PIPELINE
# ======================================================
for phase in PHASES:
    phase_df = df[df["phase"] == phase]

    data = [
        phase_df[phase_df["pipeline"] == p]["error_km"].values
        for p in pipelines
    ]

    plt.figure(figsize=(12, 6))
    plt.boxplot(data, labels=pipelines, showfliers=True)
    plt.ylabel("Haversine Error (km)")
    plt.title(f"Haversine Error Distribution — {phase}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# ======================================================
# CONFIG
# ======================================================
RESULTS_DIR = "../data/CDG-FCO/RESULTS"
PHASES = ["take_off", "cruising", "landing"]
PHASE_LABELS = ["Take-off", "Cruising", "Landing"]

PIPELINES = [
    "KALMAN",
    "ARIMA",
    "LSTM",
    "BILSTM",
    "MEAN_SIMILAR_FLIGHT",
    "RAG"
]

COLORS = {
    "KALMAN": "#1f77b4",
    "LSTM": "#ff7f0e",
    "BILSTM": "#2ca02c",
    "ARIMA": "#d62728",
    "MEAN_SIMILAR_FLIGHT": "#9467bd",
    "RAG": "#f1ff00"  # highlight
}


# ======================================================
# LOAD RESULTS
# ======================================================
results = {}

for fname in os.listdir(RESULTS_DIR):
    if not fname.endswith(".json"):
        continue

    with open(os.path.join(RESULTS_DIR, fname), "r", encoding="utf-8") as f:
        data = json.load(f)

    for method, values in data.items():
        results[method] = values


# ======================================================
# COMPUTE MEANS
# ======================================================
means = {
    method: [
        np.mean(results[method][phase])
        for phase in PHASES
    ]
    for method in PIPELINES
}


# ======================================================
# BAR PLOT (same design)
# ======================================================
x = np.arange(len(PHASES))
bar_width = 0.12

plt.figure(figsize=(14, 7))

for i, method in enumerate(PIPELINES):
    if method == "RAG":
        label = "TrajRAG (ours)"
    else:
        label = method
    offset = (i - len(PIPELINES) / 2) * bar_width + bar_width / 2
    bars = plt.bar(
        x + offset,
        means[method],
        bar_width,
        label=label,
        color=COLORS[method],
        edgecolor="black" if method == "RAG" else None,
        linewidth=2 if method == "RAG" else 0
    )

    # Value labels
    for bar in bars:
        h = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            h + 1,
            f"{h:.1f}",
            ha="center",
            va="bottom",
            fontsize=10,
            color="red" if method == "RAG" else "black",
            fontweight="bold" if method == "RAG" else "normal"
        )


# ======================================================
# STYLE
# ======================================================
plt.xticks(x, PHASE_LABELS, fontsize=12)
plt.ylabel("Mean Haversine Error (km)", fontsize=12)
plt.title(
    "Mean Haversine Error by Method and Flight Phase",
    fontsize=14
)

plt.legend(title="Method", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
import os
import json
import numpy as np
import pandas as pd

# ======================================================
# CONFIG
# ======================================================
RESULTS_DIR = "../data/CDG-FCO/RESULTS"
PHASES = ["take_off", "cruising", "landing"]

PIPELINES = [
    "KALMAN",
    "ARIMA",
    "LSTM",
    "BILSTM",
    "MEAN_SIMILAR_FLIGHT",
    "RAG"
]


# ======================================================
# LOAD RESULTS
# ======================================================
results = {}

for fname in os.listdir(RESULTS_DIR):
    if not fname.endswith(".json"):
        continue

    with open(os.path.join(RESULTS_DIR, fname), "r", encoding="utf-8") as f:
        data = json.load(f)

    for method, values in data.items():
        results[method] = values


# ======================================================
# COMPUTE STATISTICS
# ======================================================
rows = []

for phase in PHASES:
    for method in PIPELINES:
        values = np.array(results[method][phase])

        row = {
            "Phase": phase,
            "Method": method,
            "Mean": np.mean(values),
            "Median": np.median(values),
            "Std": np.std(values),
            "Min": np.min(values),
            "Q1": np.percentile(values, 25),
            "Q3": np.percentile(values, 75),
            "Max": np.max(values),
        }

        rows.append(row)

df = pd.DataFrame(rows).round(2)


# ======================================================
# PRETTY PRINT (RAG IN BOLD)
# ======================================================
def bold_if_rag(value, is_rag):
    return f"**{value}**" if is_rag else f"{value}"

pretty_rows = []

for _, row in df.iterrows():
    is_rag = row["Method"] == "RAG"
    pretty_rows.append({
        "Phase": row["Phase"],
        "Method": row["Method"],
        "Mean": bold_if_rag(row["Mean"], is_rag),
        "Median": bold_if_rag(row["Median"], is_rag),
        "Std": bold_if_rag(row["Std"], is_rag),
        "Min": bold_if_rag(row["Min"], is_rag),
        "Q1": bold_if_rag(row["Q1"], is_rag),
        "Q3": bold_if_rag(row["Q3"], is_rag),
        "Max": bold_if_rag(row["Max"], is_rag),
    })

pretty_df = pd.DataFrame(pretty_rows)

print("\n===== BOXPLOT STATISTICS TABLE (RAG IN BOLD) =====\n")
print(pretty_df.to_string(index=False))


# ======================================================
# OPTIONAL: SAVE RAW TABLE FOR LATEX / WORD
# ======================================================
df.to_csv("../data/CDG-FCO/RESULTS/BOXPLOT_STATS_ALL_METHODS.csv", index=False)
